In [ ]:
import ee
import geemap
import pandas as pd
import geopandas as gpd

In [ ]:
!pip install geojson

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project="groundwater-potential-496220")

In [ ]:
from google.colab import files
upload=files.upload()

Saving Final_Punjab_And_Potohar_Training_Data.csv to Final_Punjab_And_Potohar_Training_Data.csv


In [ ]:
df=pd.read_csv("Final_Punjab_And_Potohar_Training_Data.csv")

In [ ]:
df.head()


,LONG,LAT,WL_MBGL,geometry,potential
0,73.08917,30.47000,6.127041,POINT (73.08917 30.47),1
1,73.00361,30.61306,7.460307,POINT (73.00361 30.61306),1
2,72.80083,30.30500,7.578483,POINT (72.80083 30.305),1
3,72.85556,30.27167,8.816340,POINT (72.85556 30.27167),1
4,72.99472,30.36000,10.818571,POINT (72.99472 30.36),1


In [ ]:
points=geemap.pandas_to_ee(df,latitude="LAT",longitude="LONG")

In [ ]:
dataset = ee.Image('NASA/NASADEM_HGT/001')



In [ ]:
dataset

In [ ]:
elevation=dataset.select("elevation")
slope=ee.Terrain.slope(elevation)

In [ ]:
print([elevation,slope])

[<ee.image.Image object at 0x7bce9963fd10>, <ee.image.Image object at 0x7bce99646ba0>]


In [ ]:
neighborhood= elevation.focalMean(300,"circle","meters")

In [ ]:
tpi=elevation.subtract(neighborhood).rename("tpi")

In [ ]:
tpi

In [ ]:
soil = ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02") \
    .select('b0')

In [ ]:
soil_dataset=ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02")

In [ ]:
soil_dataset

In [ ]:
soil_surface=soil_dataset.select("b0").rename("soil_surface")   #I selected the surface of the soil as it is the primary path to infiltration

In [ ]:
soil_subsurface=soil_dataset.select("b30").rename("soil_subsurface")  #30cm too since 0cm is prome to disturbance, 30cm is deep enough and safe from disturbances.

In [ ]:
soil_surface

In [ ]:
soil_subsurface

In [ ]:
stack=ee.Image.cat([elevation, slope, tpi, soil_surface, soil_subsurface])

In [ ]:
stack

In [ ]:
Extracted_features = stack.sampleRegions(
    collection=points,
    properties=['LAT','LONG','potential', 'WL_MBGL'],
    scale=30
)


In [ ]:
points

In [ ]:
Extracted_features

In [ ]:
final_df = geemap.ee_to_df(Extracted_features)


In [ ]:
final_df.head()

,LAT,LONG,WL_MBGL,elevation,potential,slope,soil_subsurface,soil_surface,tpi
0,30.47000,73.08917,6.127041,157,1,0.000000,7,7,-0.889590
1,30.61306,73.00361,7.460307,170,1,1.077566,7,7,3.309148
2,30.30500,72.80083,7.578483,152,1,2.147576,7,7,1.037855
3,30.27167,72.85556,8.816340,150,1,2.142427,7,7,-0.138801
4,30.36000,72.99472,10.818571,156,1,0.927410,7,7,-0.441640


In [ ]:
final_df["soil_surface"].value_counts()

,count
soil_surface,
7,2233
4,249
6,176
9,59


In [ ]:
final_df["soil_subsurface"].value_counts()

,count
soil_subsurface,
7,1881
4,610
6,190
9,35
8,1


In [ ]:
final_df.to_csv("Groundwater_dataset.csv",index=False)

In [ ]:


print(f"Earth Engine (ee): {ee.__version__}")
print(f"Geemap:           {geemap.__version__}")
print(f"Pandas:           {pd.__version__}")
print(f"GeoPandas:        {gpd.__version__}")

Earth Engine (ee): 1.7.32
Geemap:           0.38.2
Pandas:           2.2.2
GeoPandas:        1.1.3
